Here, we use the GINCO and EN-GINCO datasets that have been fully annotated by 2 annotators. We apply the GPT-4o, GPT-5 and Gemini 2.5 Flash on the datasets and compare its agreement with the annotators with the agreement between the annotators.

Use the emma_main kernel

In [1]:
import pandas as pd
import json
import numpy as np

## GINCO Dataset preparation

In [6]:
file = pd.read_csv("/home/tajak/Genre-Datasets-Comparison/Creation-of-classifiers-and-cross-prediction/data-sheets-with-all-info/GINCO-MT-GINCO-keeptext-with-all-information.csv", sep="\t", index_col = 0)

file

,id,url,crawled,hard,primary_level_1,primary_level_2,primary_level_3,secondary_level_1,secondary_level_2,secondary_level_3,...,FTD_pred_on_SL,FTD_pred_on_MT,split-without-rare-categories,primary_level_4,downcast_split,CORE_main_pred_on_SL,CORE_main_pred_on_MT,CORE_sub_pred_on_SL,CORE_sub_pred_on_MT,primary_level_1_to_X-GENRE
0,3949,http://www.pomurje.si/aktualno/sport/zimska-li...,2014,False,News/Reporting,News/Reporting,News/Reporting,NaN,NaN,NaN,...,A8 (news),A8 (news),test,News/Reporting,test,Narrative,Narrative,Sports Report,Sports Report,News
1,3726,http://www.ss-sezana.si/sss/index.php?option=c...,2014,False,Information/Explanation,Information/Explanation,Information/Explanation,NaN,NaN,NaN,...,A16 (information),A16 (information),test,Information/Explanation,dev,Informational Description/Explanation,Informational Description/Explanation,Description of a Thing,Description of a Thing,Information/Explanation
2,5621,http://www.kamnik-starejsi.si/novice/144-sodel...,2014,False,Promotion of Services,Promotion of Services,Promotion,Opinion/Argumentation,Opinion/Argumentation,Opinion/Argumentation,...,A12 (promotion),A12 (promotion),train,Promotion,test,Informational Description/Explanation,Informational Description/Explanation,Description of a Thing,Description of a Thing,Promotion
3,3776,http://www.radiocelje.si/novica.php?id=13007&a...,2014,False,News/Reporting,News/Reporting,News/Reporting,NaN,NaN,NaN,...,A8 (news),A8 (news),train,News/Reporting,train,Informational Description/Explanation,Informational Description/Explanation,News Report/Blog,News Report/Blog,News
4,2102,http://www.mtv.si/novice/selena-gomez-ponudila...,2014,False,Opinionated News,Opinionated News,Opinionated News,NaN,NaN,NaN,...,A8 (news),A8 (news),test,News/Reporting,train,Narrative,Narrative,News Report/Blog,News Report/Blog,News
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,374730,http://khetanes.si/sl-si/produkti/projektne-no...,2021,False,Information/Explanation,Information/Explanation,Information/Explanation,NaN,NaN,NaN,...,A12 (promotion),A8 (news),test,Information/Explanation,test,Informational Description/Explanation,Informational Description/Explanation,Description of a Thing,Description of a Thing,Information/Explanation
998,476885,https://www.merkur.si/navigacija/nasveti/kopal...,2021,False,List of Summaries/Excerpts,List of Summaries/Excerpts,List of Summaries/Excerpts,NaN,NaN,NaN,...,A12 (promotion),A12 (promotion),train,List of Summaries/Excerpts,train,Informational Persuasion,Informational Persuasion,Description with Intent to Sell,Description with Intent to Sell,discarded
999,674213,http://www.sex2.si/category/ocene-izdelkov/,2021,False,List of Summaries/Excerpts,List of Summaries/Excerpts,List of Summaries/Excerpts,NaN,NaN,NaN,...,A12 (promotion),A12 (promotion),train,List of Summaries/Excerpts,train,Informational Persuasion,Informational Persuasion,Description with Intent to Sell,Description with Intent to Sell,discarded
1000,975590,http://www.ipsos.si/VodenjeVIZ_VI_past_dvojne_...,2021,False,Opinion/Argumentation,Opinion/Argumentation,Opinion/Argumentation,NaN,NaN,NaN,...,A1 (argumentative),A1 (argumentative),dev,Opinion/Argumentation,train,Informational Description/Explanation,Informational Description/Explanation,FAQ about Information,Question/Answer Forum,Opinion/Argumentation


In [33]:
# Keep only relevant columns
ginco = file[['id', 'hard', 'Slovene_text', 'primary_level_1', 'MT_text', 'split',]]

ginco.head(2)

,id,hard,Slovene_text,primary_level_1,MT_text,split
0,3949,False,"Šport <p/> Zimska liga malega nogometa sobota,...",News/Reporting,Sport <p/> Winter Little League Football Satur...,test
1,3726,False,JEDILNIK <p/> Iskalnik <p/> Poglavitni cilj pr...,Information/Explanation,JEDILNIK <p/> Search <p/> The main objective o...,train


In [48]:
# Open the file with annotations

ginco_ann = pd.read_csv("datasets/ginco-two-annotators-labels.txt", sep="\t")

print(ginco_ann.shape)

ginco_ann.rename(columns={"ID":"id"}, inplace=True)

ginco_ann.head(2)

(1019, 5)


,id,Taja,Mojca,Final,Final different than both initial
0,3726,Information/Explanation,Information/Explanation,Information/Explanation,no
1,5621,Promotion of Services,Information/Explanation,Promotion of Services,no


In [49]:
# Merge the annotations to the ginco dataset based on ids

ginco_final = pd.merge(left=ginco, right=ginco_ann, how="inner", on="id")

ginco_final.head(3)

,id,hard,Slovene_text,primary_level_1,MT_text,split,Taja,Mojca,Final,Final different than both initial
0,3726,False,JEDILNIK <p/> Iskalnik <p/> Poglavitni cilj pr...,Information/Explanation,JEDILNIK <p/> Search <p/> The main objective o...,train,Information/Explanation,Information/Explanation,Information/Explanation,no
1,5621,False,Projekt INNOVAge in zavod Oreli <p/> Zavod Ore...,Promotion of Services,Project INNOVAge and the Oreli Institute <p/> ...,train,Promotion of Services,Information/Explanation,Promotion of Services,no
2,3776,False,"V novembru, mesecu preprečevanja odvisnosti, b...",News/Reporting,"In November, the month of addiction prevention...",train,Invitation,News/Reporting,News/Reporting,no


In [50]:
ginco_final.shape

(996, 10)

In [59]:
# Remove instances where there is a NaN in "Taja" or "Mojca" column - this occurred because the final label to this instance was decided only at the point where the annotators discussed the differences (at the phase of assigning final annotations)
ginco_final = ginco_final.dropna(subset="Taja")
ginco_final = ginco_final.dropna(subset="Mojca")

In [65]:
# Remove texts that were assigned by at least one of the annotators to be unsuitable

ginco_final = ginco_final[ginco_final["Taja"] != "Non-textual"]
ginco_final = ginco_final[ginco_final["Mojca"] != "Multiple texts"]

ginco_final.shape

(990, 10)

In [51]:
ginco_final.columns

Index(['id', 'hard', 'Slovene_text', 'primary_level_1', 'MT_text', 'split',
       'Taja', 'Mojca', 'Final', 'Final different than both initial'],
      dtype='object')

In [44]:
# Clean the labels - correct labels if they do not appear in the label list
label_list = ['Announcement', 'Call', 'Correspondence', 'FAQ', 'Forum', 'Information/Explanation', 'Instruction', 'Interview', 'Invitation', 'Legal/Regulation', 'List of Summaries/Excerpts', 'Lyrical', 'News/Reporting', 'Opinion/Argumentation', 'Opinionated News', 'Other', 'Promotion', 'Promotion of Services', 'Promotion of a Product', 'Prose', 'Recipe', 'Research Article', 'Review', 'Script/Drama']

In [69]:
labels_to_check = list(ginco_final["Taja"].unique())

for label in labels_to_check:
	if label not in label_list:
		print(label)

In [70]:
# Check whether the final annotations agree
ginco_final.query("primary_level_1 != Final")[["Slovene_text", "primary_level_1", "Final"]]

,Slovene_text,primary_level_1,Final
3,Selena Gomez ponudila v poslušanje novi album ...,Opinionated News,News/Reporting
10,Lezbični in feministični klub v vsako vas Sobo...,Opinionated News,Opinion/Argumentation
18,PRIJETNO DRUŽENJE Z IZMENJAVO RABLJENIH OBLAČI...,Opinionated News,Opinion/Argumentation
23,Stoner bo sprejet med motociklistične legende ...,Opinionated News,News/Reporting
59,"Oče alkoholik uničuje življenje študentki, ki ...",Correspondence,Other
60,Farmacevt odgovarja <p/> Vprašanje: Pozdravlje...,Correspondence,Other
64,ODMEVI NA KNJIGO <p/> (Po e-pošti 24.06.2020 -...,Correspondence,Other
126,Pravice potnikov v letalskem prometu <p/> Ljud...,Instruction,Information/Explanation
160,"Javni razpis za izbor kulturnih programov, ki ...",Call,Information/Explanation
171,Šolska Malica <p/> PRIJAVA MALICE <p/> Na mali...,Instruction,Other


In [71]:
ginco_final.query("(primary_level_1 != Taja)&(primary_level_1 != Mojca)")[["Slovene_text", "primary_level_1", "Taja", "Mojca", "Final"]]

,Slovene_text,primary_level_1,Taja,Mojca,Final
3,Selena Gomez ponudila v poslušanje novi album ...,Opinionated News,Opinion/Argumentation,News/Reporting,News/Reporting
4,Razno <p/> Letos so Švicarji že tridesetič pri...,Promotion of a Product,Review,Information/Explanation,Promotion of a Product
10,Lezbični in feministični klub v vsako vas Sobo...,Opinionated News,Opinion/Argumentation,Opinion/Argumentation,Opinion/Argumentation
18,PRIJETNO DRUŽENJE Z IZMENJAVO RABLJENIH OBLAČI...,Opinionated News,Opinion/Argumentation,News/Reporting,Opinion/Argumentation
23,Stoner bo sprejet med motociklistične legende ...,Opinionated News,Opinion/Argumentation,News/Reporting,News/Reporting
...,...,...,...,...,...
958,"Sončne elektrarne se da pogasiti <p/> ""Sončne ...",Other,News/Reporting,News/Reporting,Other
960,34 milijardni prevzem Red Hata s strani IBM-a ...,Opinionated News,Opinion/Argumentation,News/Reporting,Opinionated News
962,Kaj ima skupnega vseh 500 najzmogljivejših sup...,Opinionated News,News/Reporting,News/Reporting,Opinionated News
963,O trgovinski vojni med ZDA in Kitajsko se zadn...,Opinionated News,Opinion/Argumentation,Opinion/Argumentation,Opinionated News


As we can see above, the label "Opinionated News" is problematic, because it was defined later in the annotation process as a reaction to instances that were between Opinion and News. For the purpose of this study, we will discard instances, annotated with this label, because the disagreement between the annotators in these cases was solely due to a blind spot in the annotation guidelines which was later solved by introduction of the label "Opinionated News".

In [101]:
ginco_final = ginco_final[ginco_final["primary_level_1"] != "Opinionated News"]
ginco_final = ginco_final[ginco_final["Taja"] != "Opinionated News"]
ginco_final = ginco_final[ginco_final["Mojca"] != "Opinionated News"]
ginco_final.shape

(847, 8)

In [82]:
ginco_final["primary_level_1"].value_counts()

primary_level_1
Information/Explanation       128
Promotion of a Product        115
News/Reporting                114
Opinion/Argumentation         112
List of Summaries/Excerpts    105
Forum                          50
Instruction                    37
Other                          33
Promotion of Services          32
Invitation                     31
Promotion                      30
Review                         17
Legal/Regulation               17
Announcement                   17
Correspondence                 16
Call                           11
Research Article                9
Interview                       8
Recipe                          6
Prose                           6
Lyrical                         3
FAQ                             3
Script/Drama                    1
Name: count, dtype: int64

Similarly, we discard instances labelled as "Correspondence" or "Call", as these labels were introduced during the annotation process and have not been present since the beginning of the annotation campaign.

In [102]:
ginco_final = ginco_final[ginco_final["primary_level_1"] != "Correspondence"]
ginco_final = ginco_final[ginco_final["primary_level_1"] != "Call"]
ginco_final = ginco_final[ginco_final["Taja"] != "Correspondence"]
ginco_final = ginco_final[ginco_final["Taja"] != "Call"]
ginco_final = ginco_final[ginco_final["Mojca"] != "Correspondence"]
ginco_final = ginco_final[ginco_final["Mojca"] != "Call"]
ginco_final.shape

(844, 8)

In [86]:
ginco_final.query("(primary_level_1 != Taja)&(primary_level_1 != Mojca)")[["Slovene_text", "primary_level_1", "Taja", "Mojca", "Final"]]

,Slovene_text,primary_level_1,Taja,Mojca,Final
4,Razno <p/> Letos so Švicarji že tridesetič pri...,Promotion of a Product,Review,Information/Explanation,Promotion of a Product
27,Ko dobimo otroka v svoje naročje je tako kot b...,Opinion/Argumentation,Promotion,Promotion of Services,Opinion/Argumentation
164,Main menu <p/> Post navigation <p/> Pogled na ...,Other,Promotion,Promotion,Other
171,Šolska Malica <p/> PRIJAVA MALICE <p/> Na mali...,Instruction,Legal/Regulation,Announcement,Other
181,V drugem delu uspešne filmske serije se bo mor...,Other,Prose,Promotion of a Product,Other
232,Govor predsednika Sveta za splošne zadeve in z...,Other,Opinion/Argumentation,Opinion/Argumentation,Other
295,Sporočila za javnost <p/> 14.12.2004 <p/> Hura...,Information/Explanation,Promotion,News/Reporting,Information/Explanation
365,Celoten »pulmo tim« Kliničnega oddelka za plju...,Other,News/Reporting,Promotion,Other
388,Naj projekt za mlade 2011 <p/> V okviru projek...,News/Reporting,Call,Promotion,News/Reporting
497,REDNI DELOVNI ČAS NAŠIH PRODAJALN <p/> Z izjem...,Promotion,Announcement,Promotion of Services,Promotion


In [96]:
# Additionally, we remove instances where the final label is different than the labels of either of the annotators - this means that in the annotation discussion phase, the annotators jointly decided to overthrow their initial decisions and decide for a completely different label. We discard these instances because we cannot say which of these labels is the gold label for the GPT model when we evaluate its performance.

ginco_final = ginco_final.query("(primary_level_1 == Taja) or (primary_level_1 == Mojca)")

print(ginco_final.shape)

ginco_final.head(5)

(848, 10)


,id,hard,Slovene_text,primary_level_1,MT_text,split,Taja,Mojca,Final,Final different than both initial
0,3726,False,JEDILNIK <p/> Iskalnik <p/> Poglavitni cilj pr...,Information/Explanation,JEDILNIK <p/> Search <p/> The main objective o...,train,Information/Explanation,Information/Explanation,Information/Explanation,no
1,5621,False,Projekt INNOVAge in zavod Oreli <p/> Zavod Ore...,Promotion of Services,Project INNOVAge and the Oreli Institute <p/> ...,train,Promotion of Services,Information/Explanation,Promotion of Services,no
2,3776,False,"V novembru, mesecu preprečevanja odvisnosti, b...",News/Reporting,"In November, the month of addiction prevention...",train,Invitation,News/Reporting,News/Reporting,no
5,6979,False,Uvajanje moderne tehnologije in sledenje hitre...,Invitation,The introduction of modern technology and keep...,train,Invitation,Invitation,Invitation,no
6,5649,False,"JEDRO: Pravilo iz 216. člena ZPP omogoča, da s...",Legal/Regulation,JUDGMENT: The rule in Article 216 of the CCP a...,dev,Legal/Regulation,Legal/Regulation,Legal/Regulation,no


In [98]:
# Discard the column "Final"

ginco_final.drop(columns=["Final", "Final different than both initial"], inplace=True)
ginco_final.head(2)

,id,hard,Slovene_text,primary_level_1,MT_text,split,Taja,Mojca
0,3726,False,JEDILNIK <p/> Iskalnik <p/> Poglavitni cilj pr...,Information/Explanation,JEDILNIK <p/> Search <p/> The main objective o...,train,Information/Explanation,Information/Explanation
1,5621,False,Projekt INNOVAge in zavod Oreli <p/> Zavod Ore...,Promotion of Services,Project INNOVAge and the Oreli Institute <p/> ...,train,Promotion of Services,Information/Explanation


In [105]:
labels = ['Information/Explanation',
 'Promotion of Services',
 'News/Reporting',
 'Invitation',
 'Legal/Regulation',
 'Promotion of a Product',
 'Opinion/Argumentation',
 'Announcement',
 'Forum',
 'FAQ',
 'Instruction',
 'Research Article',
 'List of Summaries/Excerpts',
 'Promotion',
 'Lyrical',
 'Recipe',
 'Review',
 'Prose',
 'Other',
 'Interview',
 'Script/Drama']

In [106]:
len(labels)

21

In [109]:
print(len(ginco_final["Mojca"].unique()))
print(list(ginco_final["Mojca"].unique()))

for i in ginco_final["Mojca"].unique():
	if i not in labels:
		print(i)

21
['Information/Explanation', 'News/Reporting', 'Invitation', 'Legal/Regulation', 'Opinion/Argumentation', 'Announcement', 'Forum', 'Promotion of a Product', 'FAQ', 'Instruction', 'Research Article', 'List of Summaries/Excerpts', 'Other', 'Promotion of Services', 'Review', 'Promotion', 'Lyrical', 'Recipe', 'Prose', 'Interview', 'Script/Drama']


In [111]:
ginco_final.primary_level_1.value_counts()

primary_level_1
Information/Explanation       124
Promotion of a Product        114
News/Reporting                113
Opinion/Argumentation         111
List of Summaries/Excerpts    104
Forum                          50
Instruction                    36
Promotion of Services          31
Invitation                     31
Promotion                      25
Other                          18
Announcement                   17
Legal/Regulation               17
Review                         17
Research Article                9
Interview                       8
Recipe                          6
Prose                           6
FAQ                             3
Lyrical                         3
Script/Drama                    1
Name: count, dtype: int64

In [112]:
# Save the created dataset

ginco_final.to_json("datasets/final_ginco_with_annotators_labels.jsonl", orient="records", lines=True)

In [2]:
ginco_final = pd.read_json("datasets/final_ginco_with_annotators_labels.jsonl", orient="records", lines=True)
ginco_final

,id,hard,Slovene_text,primary_level_1,MT_text,split,Taja,Mojca
0,3726,False,JEDILNIK <p/> Iskalnik <p/> Poglavitni cilj pr...,Information/Explanation,JEDILNIK <p/> Search <p/> The main objective o...,train,Information/Explanation,Information/Explanation
1,5621,False,Projekt INNOVAge in zavod Oreli <p/> Zavod Ore...,Promotion of Services,Project INNOVAge and the Oreli Institute <p/> ...,train,Promotion of Services,Information/Explanation
2,3776,False,"V novembru, mesecu preprečevanja odvisnosti, b...",News/Reporting,"In November, the month of addiction prevention...",train,Invitation,News/Reporting
3,6979,False,Uvajanje moderne tehnologije in sledenje hitre...,Invitation,The introduction of modern technology and keep...,train,Invitation,Invitation
4,5649,False,"JEDRO: Pravilo iz 216. člena ZPP omogoča, da s...",Legal/Regulation,JUDGMENT: The rule in Article 216 of the CCP a...,dev,Legal/Regulation,Legal/Regulation
...,...,...,...,...,...,...,...,...
839,374730,False,Projektne novine <p/> Promocijski projektni ča...,Information/Explanation,Project News <p/> Promotional project newspape...,train,Information/Explanation,Information/Explanation
840,476885,False,V raznoliki ponudbi tušev izberite popolno raz...,List of Summaries/Excerpts,Choose the perfect shower to match your taste ...,train,List of Summaries/Excerpts,List of Summaries/Excerpts
841,674213,False,"O izdelku Za znamko Dame stojita dve ženski, z...",List of Summaries/Excerpts,About the product There are two women behind t...,train,List of Summaries/Excerpts,List of Summaries/Excerpts
842,975590,False,Razprava pogosto potegne na plano najprej tist...,Opinion/Argumentation,The debate often brings to the surface first t...,train,Opinion/Argumentation,Opinion/Argumentation


In [3]:
ginco_final.shape

(844, 8)

In [4]:
# Now map the X-GENRE labels to GINCO labels
mapping = {'FAQ': 'discarded', 'List of Summaries/Excerpts': 'discarded', 'Forum': 'Forum', 'Information/Explanation': 'Information/Explanation', 'Research Article': 'Information/Explanation', 'Instruction': 'Instruction', 'Recipe': 'Instruction', 'Legal/Regulation': 'Legal', 'Announcement': 'News', 'News/Reporting': 'News', 'Opinionated News': 'News', 'Opinion/Argumentation': 'Opinion/Argumentation', 'Review': 'Opinion/Argumentation', 'Call': 'Other', 'Correspondence': 'Other', 'Interview': 'Other', 'Other': 'Other', 'Script/Drama': 'Other', 'Invitation': 'Promotion', 'Promotion': 'Promotion', 'Promotion of a Product': 'Promotion', 'Promotion of Services': 'Promotion', 'Lyrical': 'Prose/Lyrical', 'Prose': 'Prose/Lyrical'}


def map_to_xgenre(value):
	if value in list(mapping.keys()):
		new_value = mapping[value]
	else:
		new_value = value
	return new_value


In [5]:
ginco_final["Ann1"] = ginco_final["Mojca"].apply(map_to_xgenre)
ginco_final["Ann2"] = ginco_final["Taja"].apply(map_to_xgenre)
ginco_final["Gold_X-GENRE"] = ginco_final["primary_level_1"].apply(map_to_xgenre)

ginco_final[["primary_level_1", "Gold_X-GENRE", "Mojca", "Ann1", "Taja", "Ann2"]].head(5)

,primary_level_1,Gold_X-GENRE,Mojca,Ann1,Taja,Ann2
0,Information/Explanation,Information/Explanation,Information/Explanation,Information/Explanation,Information/Explanation,Information/Explanation
1,Promotion of Services,Promotion,Information/Explanation,Information/Explanation,Promotion of Services,Promotion
2,News/Reporting,News,News/Reporting,News,Invitation,Promotion
3,Invitation,Promotion,Invitation,Promotion,Invitation,Promotion
4,Legal/Regulation,Legal,Legal/Regulation,Legal,Legal/Regulation,Legal


In [6]:
# Remove instances where any of the annotators (or the final label) assigned the label that is now "discarded"

ginco_final = ginco_final[ginco_final["Gold_X-GENRE"] != "discarded"]
ginco_final = ginco_final[ginco_final["Ann1"] != "discarded"]
ginco_final = ginco_final[ginco_final["Ann2"] != "discarded"]

ginco_final.shape

(725, 11)

In [9]:
ginco_final["Ann2"].value_counts()

Ann2
Promotion                  197
Opinion/Argumentation      129
Information/Explanation    128
News                       123
Forum                       49
Instruction                 46
Other                       27
Legal                       17
Prose/Lyrical                9
Name: count, dtype: int64

In [10]:
# Save the dataset
ginco_final.to_json("datasets/final_ginco_with_annotators_labels.jsonl", orient="records", lines=True)


## EN-GINCO Dataset Preparation

In [1]:
import pandas as pd

file = pd.read_csv("datasets/EN-GINCO-suitable.csv", sep="\t", index_col = 0)

file

,id,primary,secondary,tertiary,hard,title,url,crawled,topic,se-genre,text,X-GENRE
0,11.0,Information/Explanation,Promotion,NaN,False,Naval & Military History,http://kbismarck.org/,2019-11-30,society,none,Welcome to KBismarck.org! This is a community ...,Information/Explanation
1,3214578.0,News/Reporting,NaN,Opinionated News,False,Why graft thrives in postconflict zones - CSMo...,https://www.csmonitor.com/2005/0317/p06s01-wog...,2019-11-30,none,news,Why graft thrives in postconflict zones <p> A ...,News
2,9423821.0,Invitation,NaN,NaN,False,KES-InMed-16 : Social Events,http://inmed-16.kesinternational.org/social.php,2019-11-30,none,none,Social Trip <p> On the evening of Wednesday 15...,Promotion
3,12814029.0,Promotion of a Product,NaN,NaN,False,Caple Wok Burner Outer Cap | www.4caple.co.uk,https://www.4caple.co.uk/wok-burner-outer-cap/...,2019-11-30,none,none,If the burner cap on your hob is not looking l...,Promotion
4,12814029.1,List of Summaries/Excerpts,NaN,NaN,False,Caple Wok Burner Outer Cap | www.4caple.co.uk,https://www.4caple.co.uk/wok-burner-outer-cap/...,2019-11-30,none,none,Diameter: 61 (mm) If the burner cap on your ho...,discarded
...,...,...,...,...,...,...,...,...,...,...,...,...
295,80843352.0,Promotion,NaN,NaN,False,&apos;50 things to do before you&apos;re 11¾&a...,https://www.nationaltrust.org.uk/canons-ashby/...,2019-12-04,none,none,There's lots of outdoor adventures to enjoy at...,Promotion
296,84840780.0,Information/Explanation,NaN,Opinion/Argumentation,True,New Radnor Boroughs | History of Parliament On...,https://www.historyofparliamentonline.org/volu...,2019-12-04,none,none,Although New Radnor itself exceeded in size an...,Information/Explanation
297,88841804.0,Opinion/Argumentation,Information/Explanation,NaN,True,A Call For All Member States of the Human Righ...,http://www.globaljusticecenter.net/press-cente...,2019-12-04,none,none,Press Releases <p> A Call For All Member State...,Opinion/Argumentation
298,92842404.0,Promotion,NaN,NaN,False,Trendelburg - Experience enchanting nature in ...,http://www.deutsche-maerchenstrasse.com/en/you...,2019-12-05,none,none,Trendelburg - Experience enchanting nature in ...,Promotion


In [2]:
file.columns

Index(['id', 'primary', 'secondary', 'tertiary', 'hard', 'title', 'url',
       'crawled', 'topic', 'se-genre', 'text', 'X-GENRE'],
      dtype='object')

In [3]:
# Discard texts with "discarded" as X-GENRE label
file = file[file["X-GENRE"] != "discarded"]

print(file.shape)

file['X-GENRE'].value_counts()

(272, 12)


X-GENRE
Information/Explanation    64
Promotion                  63
Opinion/Argumentation      50
News                       46
Forum                      20
Other                      14
Instruction                10
Prose/Lyrical               4
Legal                       1
Name: count, dtype: int64

In [4]:
# Keep only relevant columns
en_ginco = file[['id', 'text', 'X-GENRE']]
en_ginco.rename(columns={"X-GENRE":'labels', 'id': 'ID'}, inplace=True)

en_ginco.head(2)

/tmp/ipykernel_2146992/1060371974.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  en_ginco.rename(columns={"X-GENRE":'labels', 'id': 'ID'}, inplace=True)


,ID,text,labels
0,11.0,Welcome to KBismarck.org! This is a community ...,Information/Explanation
1,3214578.0,Why graft thrives in postconflict zones <p> A ...,News


In [5]:
en_ginco.shape

(272, 3)

In [6]:
# Open the file with annotations

en_ginco_ann = pd.read_csv("datasets/en-ginco-two-annotators-labels.txt", sep="\t")

print(en_ginco_ann.shape)

en_ginco_ann.head(2)

(301, 5)


,ID,Taja,Mojca,Final,Final different than both initial
0,11.0,Information/Explanation,Promotion,Information/Explanation,no
1,3214578.0,News/Reporting,News/Reporting,News/Reporting,no


In [7]:
# Merge the annotations to the en_ginco dataset based on ids

en_ginco_final = pd.merge(left=en_ginco, right=en_ginco_ann, how="inner", on="ID")

en_ginco_final.head(3)

,ID,text,labels,Taja,Mojca,Final,Final different than both initial
0,11.0,Welcome to KBismarck.org! This is a community ...,Information/Explanation,Information/Explanation,Promotion,Information/Explanation,no
1,3214578.0,Why graft thrives in postconflict zones <p> A ...,News,News/Reporting,News/Reporting,News/Reporting,no
2,9423821.0,Social Trip <p> On the evening of Wednesday 15...,Promotion,Promotion of a Product,Invitation,Invitation,no


In [8]:
en_ginco_final.shape

(271, 7)

In [9]:
# Remove instances where there is a NaN in "Taja" or "Mojca" column - this occurred because the final label to this instance was decided only at the point where the annotators discussed the differences (at the phase of assigning final annotations)
en_ginco_final = en_ginco_final.dropna(subset="Taja")
en_ginco_final = en_ginco_final.dropna(subset="Mojca")
en_ginco_final.shape

(268, 7)

In [10]:
# Remove texts that were assigned by at least one of the annotators to be unsuitable

en_ginco_final = en_ginco_final[en_ginco_final["Taja"] != "Non-textual"]
en_ginco_final = en_ginco_final[en_ginco_final["Mojca"] != "Multiple texts"]

en_ginco_final.shape

(268, 7)

In [11]:
# Clean the labels - correct labels if they do not appear in the label list
label_list = ['Announcement', 'Call', 'Correspondence', 'FAQ', 'Forum', 'Information/Explanation', 'Instruction', 'Interview', 'Invitation', 'Legal/Regulation', 'List of Summaries/Excerpts', 'Lyrical', 'News/Reporting', 'Opinion/Argumentation', 'Opinionated News', 'Other', 'Promotion', 'Promotion of Services', 'Promotion of a Product', 'Prose', 'Recipe', 'Research Article', 'Review', 'Script/Drama']

In [13]:
labels_to_check = list(en_ginco_final["Mojca"].unique())

for label in labels_to_check:
	if label not in label_list:
		print(label)

In [14]:
en_ginco_final.columns

Index(['ID', 'text', 'labels', 'Taja', 'Mojca', 'Final',
       'Final different than both initial'],
      dtype='object')

In [16]:
en_ginco_final["Final different than both initial"].value_counts()

Final different than both initial
no     261
yes      7
Name: count, dtype: int64

In [17]:
# Remove instances where the final label is different than any of the initial labels provided by the two annotators - this means that in the annotation discussion phase, the annotators jointly decided to overthrow their initial decisions and decide for a completely different label.
en_ginco_final = en_ginco_final[en_ginco_final["Final different than both initial"] == "no"]
en_ginco_final.shape

(261, 7)

In [19]:
en_ginco_final.query("(Final != Taja)&(Final != Mojca)")[["text", "Final", "Taja", "Mojca"]]

,text,Final,Taja,Mojca


In [21]:
en_ginco_final.head(2)

,ID,text,labels,Taja,Mojca,Final,Final different than both initial
0,11.0,Welcome to KBismarck.org! This is a community ...,Information/Explanation,Information/Explanation,Promotion,Information/Explanation,no
1,3214578.0,Why graft thrives in postconflict zones <p> A ...,News,News/Reporting,News/Reporting,News/Reporting,no


In [25]:
# Discard certain columns

en_ginco_final.drop(columns=["Final different than both initial"], inplace=True)
en_ginco_final.rename(columns={"Final": "primary_level_1", "labels": "Gold_X-GENRE" }, inplace=True)
en_ginco_final.head(2)

,ID,text,Gold_X-GENRE,Taja,Mojca,primary_level_1
0,11.0,Welcome to KBismarck.org! This is a community ...,Information/Explanation,Information/Explanation,Promotion,Information/Explanation
1,3214578.0,Why graft thrives in postconflict zones <p> A ...,News,News/Reporting,News/Reporting,News/Reporting


In [23]:
# Now map the X-GENRE labels to GINCO labels
mapping = {'FAQ': 'discarded', 'List of Summaries/Excerpts': 'discarded', 'Forum': 'Forum', 'Information/Explanation': 'Information/Explanation', 'Research Article': 'Information/Explanation', 'Instruction': 'Instruction', 'Recipe': 'Instruction', 'Legal/Regulation': 'Legal', 'Announcement': 'News', 'News/Reporting': 'News', 'Opinionated News': 'News', 'Opinion/Argumentation': 'Opinion/Argumentation', 'Review': 'Opinion/Argumentation', 'Call': 'Other', 'Correspondence': 'Other', 'Interview': 'Other', 'Other': 'Other', 'Script/Drama': 'Other', 'Invitation': 'Promotion', 'Promotion': 'Promotion', 'Promotion of a Product': 'Promotion', 'Promotion of Services': 'Promotion', 'Lyrical': 'Prose/Lyrical', 'Prose': 'Prose/Lyrical'}


def map_to_xgenre(value):
	if value in list(mapping.keys()):
		new_value = mapping[value]
	else:
		new_value = value
	return new_value


In [27]:
en_ginco_final["Ann1"] = en_ginco_final["Mojca"].apply(map_to_xgenre)
en_ginco_final["Ann2"] = en_ginco_final["Taja"].apply(map_to_xgenre)

en_ginco_final[["primary_level_1", "Gold_X-GENRE", "Mojca", "Ann1", "Taja", "Ann2"]].head(5)

,primary_level_1,Gold_X-GENRE,Mojca,Ann1,Taja,Ann2
0,Information/Explanation,Information/Explanation,Promotion,Promotion,Information/Explanation,Information/Explanation
1,News/Reporting,News,News/Reporting,News,News/Reporting,News
2,Invitation,Promotion,Invitation,Promotion,Promotion of a Product,Promotion
3,Promotion of a Product,Promotion,Promotion of a Product,Promotion,Promotion of a Product,Promotion
4,Opinion/Argumentation,Opinion/Argumentation,Information/Explanation,Information/Explanation,Opinion/Argumentation,Opinion/Argumentation


In [28]:
# Remove instances where any of the annotators (or the final label) assigned the label that is now "discarded"

en_ginco_final = en_ginco_final[en_ginco_final["Gold_X-GENRE"] != "discarded"]
en_ginco_final = en_ginco_final[en_ginco_final["Ann1"] != "discarded"]
en_ginco_final = en_ginco_final[en_ginco_final["Ann2"] != "discarded"]

en_ginco_final.shape

(253, 8)

In [31]:
en_ginco_final["Gold_X-GENRE"].value_counts()

Gold_X-GENRE
Information/Explanation    62
Promotion                  57
Opinion/Argumentation      48
News                       45
Forum                      18
Instruction                10
Other                       9
Prose/Lyrical               3
Legal                       1
Name: count, dtype: int64

In [32]:
# Save the dataset
en_ginco_final.to_json("datasets/final_EN-GINCO_with_annotators_labels.jsonl", orient="records", lines=True)
